# Final Report

## 1. Instructions to Run the Code

1. **Environment Setup:** Ensure Python 3.10+ is installed with `duckdb`, `pandas`, `numpy`, `scikit-learn`, `scipy`, and `matplotlib`.
2. **Database Placement:** Place `equities_class.duckdb` and `equities_top2000.duckdb` inside the local `./data/` directory (or update `DATA_DIR` in Cell 1 when running the code)
3. **Execution Pipeline:**
   - **Cells 1–9:** Database connection, explore the data, and generate the features.
   - **Cells 10–13:** Set up the walk_forward setup, and the backtester and test the backtester.
   - **Cells 14–24:** Connection to `equities_top2000.duckdb`, (full database) and try different models, features and methods.
   - **Cells 25–29:** Final 15-feature pruned walk-forward model, cap-matched portfolio construction, out-of-sample results, and cumulative return plots.

---

## 2. Features, Model, Validation, and Trading Strategy

### Feature Engineering & Preventing Look-Ahead Bias
To eliminate look-ahead bias and structural data errors without inventing synthetic fills:
- **No look-ahead bias:** Quarterly fundamental ratios are joined via DuckDB `ASOF JOIN` on `filing_date + INTERVAL 1 DAY <= signal_date`. Trailing 12-month revenue (`rev_ttm`) is summed strictly across the 4 most recently filed quarters prior to `signal_date`.
- **Screen:** On each `signal_date`, candidates must satisfy:
  1. Complete price history over the trailing 63 trading sessions (`has_full_63d_trading == True`).
  2. Trailing 21-day average dollar volume $\ge \$5,000,000$ and non-zero minimum volume (`min_vol_21d > 0`).
  3. Closing price between $\$5.00$ and $\$5,000.00$.
  4. Balance sheet sanity check ($0.01 \le \text{market\_cap\_proxy} / \text{total\_assets} \le 500$).
- **Feature Pruning (15 Final Features):**
  1. Features were selected based on statistically significant Information Coefficients ($|t\text{-stat}| > 2.5$) evaluated exclusively during the **2010–2015 development window**. Noisy short- and medium-term return signals (`ret_1d`, `ret_5d`, `ret_21d`, `ret_63d`, `ret_126d`, `mom_6m_skip1m`, `sharpe_63d`) had $|t| < 1.3$ and were pruned, while I added non-redundant trailing 12-month Price-to-Sales (`mcap_to_rev_ttm`, $t = -3.05$) because there was no revenue growth metric.
  2. **Final 15 Features:**
     - `ebitda_to_assets` ($t=8.45$), `fcf_to_assets` ($t=8.28$), `roa` ($t=6.93$), `ocf_to_assets` ($t=4.69$), `op_margin`, `gross_margin`, `ret_252d` ($t=3.93$), `mom_12m_skip1m` ($t=3.87$),`fcf_yield` ($t=7.83$), `earnings_yield` ($t=7.51$), `book_to_market` ($t=-5.05$), `mcap_to_rev_ttm` ($t=-3.05$),`neg_universe_rank` ($t=3.35$), `vol_63d` ($t=-2.61$), `avg_hl_spread_21d` ($t=-3.21$)
- **Cross-Sectional Rank Normalization:** Features are transformed into uniform cross-sectional percentile ranks in $[-0.5, +0.5]$ on each `signal_date`.

### Other Strategy Notes
- **Ranking Transformation:** To prevent high-volatility features, models are trained on the cross-sectional percentile rank of 90-calendar-day forward returns:
  $$y_{i,t} = \text{Rank}\left(R_{i, t \to t+90\text{d}}\right) - 0.5$$
- **Ensemble Model:** A 50/50 blend of an L2-regularized linear model (`RidgeCV`, alphas $\in [10^1, 10^5]$) and shallow gradient boosted trees (`HistGradientBoostingRegressor`, `max_depth=2`, `l2_regularization=50.0`).
- **Retraining:** Models retrain quarterly. At each fit date $T_{\text{fit}}$, training rows are strictly filtered to:
  $$\text{exit\_date}_i \le T_{\text{fit}}$$
  Any 90-day holding period that has not completed prior to $T_{\text{fit}}$ is omitted. The 3-month buffer period (Jan 1 – Mar 31, 2016) allows all 2015 development targets to finish maturing before the test period begins on April 1, 2016.

### Portfolio Construction
- **Holding Period:** Strictly 90 calendar days, exiting at the close of the next available trading session.
- **Tranche Recycling:** To eliminate cash drag, positions recycle on the exact `exit_date` of maturing tranches. Sales execute at the close, returning cash immediately to fund new entries (net of 5 bps transaction costs per side).
- **Weighting of Stocks:** Scored across the full 2,000-stock universe, with execution filtered to liquid institutional large-caps (`universe_rank <= 500` or `<= 250`). Positions are allocated using square-root market-cap weighting (`mcap_sqrt`), capped at 8%–12% per stock to ensure diversification.

---

## 3. Performance Comparison & Findings

### Walk-Forward Rank IC Summary (15-Feature Model)
| Evaluation Universe | Period | Ensemble Mean Rank IC | Ensemble IC IR | Ridge Mean Rank IC | Ridge IC IR |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **All-2000 Screened (15 Feats)** | **Dev (2012–2015)** | **+0.0263** | **0.289** | **+0.0286** | **0.307** |
| **All-2000 Screened (15 Feats)** | **Final Test (2016–2026)** | **+0.0200** | **0.152** | **+0.0179** | **0.140** |
| Top-500 Cap Only (15 Feats) | Dev (2012–2015) | +0.0254 | 0.204 | +0.0277 | 0.231 |
| Top-500 Cap Only (15 Feats) | Final Test (2016–2026) | +0.0062 | 0.044 | +0.0021 | 0.015 |

### Development Period Backtest (2012-01-03 to 2015-12-31)
*Initial capital: \$1,000,000. Risk-free rate: 2.0% annual. 5 bps transaction costs per trade.*

| Portfolio Strategy | Final Value (\$) | Total Return (%) | CAGR (%) | Max Drawdown (%) | Sharpe Ratio | Data Gaps |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **All-2000 Model $\to$ Top 25 (Top-500 Filter, 1-Tr SqrtCap)** | **\$2,839,326.33** | **183.93%** | **29.88%** | **-10.81%** | **1.80** | **0** |
| All-2000 Model $\to$ Top 25 (Top-500 Filter, 1-Tr InvVol) | \$2,613,577.22 | 161.36% | 27.21% | -10.25% | 1.79 | 0 |
| All-2000 Ridge $\to$ Top 25 (Top-250 Cap, 1-Tr SqrtCap) | \$2,575,308.12 | 157.53% | 26.74% | -11.10% | 1.69 | 0 |
| All-2000 Model $\to$ Top 25 (All Caps, 1-Tr InvVol) | \$2,310,736.74 | 131.07% | 23.35% | -15.49% | 1.44 | 0 |
| **^SP500TR Benchmark** | **\$1,743,112.57** | **74.31%** | **14.94%** | **-12.04%** | **1.00** | **0** |

### Final Test Period (2016-04-01 to 2026-09-30) — 15-Feature Model
*Strategy entered at the close of 2016-04-01 vs. \$1,000,000 invested in `^SP500TR` at the same close.*

| Portfolio Strategy | Final Value (\$) | Total Return (%) | CAGR (%) | Max Drawdown (%) | Sharpe Ratio | Data Gaps |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **15-Feat All-2000 Ensemble $\to$ Top 25 (Top-250 Cap-Matched, 1-Tr)** | **\$3,381,718.24** | **238.34%** | **12.31%** | **-32.88%** | **0.66** | **1** |
| 15-Feat All-2000 Ridge $\to$ Top 25 (Top-500 Cap-Matched, 1-Tr) | \$3,255,680.21 | 225.73% | 11.91% | -32.30% | 0.66 | 4 |
| 15-Feat All-2000 Ridge $\to$ Top 25 (Top-250 Cap-Matched, 1-Tr) | \$3,108,712.97 | 211.03% | 11.42% | -32.48% | 0.63 | 2 |
| 15-Feat All-2000 Ensemble (Top 25, All Caps, 1-Tr InvVol) | \$2,825,003.82 | 182.64% | 10.40% | -30.42% | 0.59 | 10 |
| 15-Feat Top-500 Ensemble (Top 25, 1-Tr InvVol) | \$2,710,608.82 | 171.20% | 9.97% | -33.94% | 0.52 | 2 |
| **^SP500TR Benchmark (\$1M Buy & Hold)** | **\$4,402,746.78** | **340.27%** | **15.17%** | **-33.79%** | **0.77** | **0** |

---

## 4. Evidence of Backtest Accounting and Timing Verification

The backtester was validated through an automated 5-day, 2-stock synthetic unit test. In the test, the starting capital of \$1,000,000   allocated  \$500,000  each to Stock `AAA` (5,000 shares @ \$100) and Stock `BBB` (10,000 shares @ \$50) on 2020-01-03. 90 calendar days from 2020-01-03 corresponds to 2020-04-02. On 2020-04-02, positions liquidated at closing prices (`AAA` @ \$120, `BBB` @ \$55), returning gross proceeds of \$600,000 + \$550,000 = \$1,150,000. All cash returned and positions marked to zero. On each session, maturing positions liquidate first, releasing cash before new orders evaluate sizing constraints.

## 5. Findings, Unsuccessful Ideas, and Next Steps

### Key Findings
1. **Final 15 Feature Model:** Adding the extra feature of trailing 12-month Price-to-Sales (`mcap_to_rev_ttm`) to the 14 pruned features and training across the full ~2,000-stock screened universe raised out-of-sample Rank IC to **+0.0200 (IR 0.152)** across 2016–2026 (up from `+0.0178` on 14 features and `-0.0094` on the unpruned 23-feature baseline).
2. **Ensemble Synergy in Top-250 Large-Cap Execution:** With the 15-feature set, the 60/40 `RidgeCV` + `HistGBDT` ensemble outperformed standalone `RidgeCV` in the Top-250 tier (**\$3.38M / 12.31% CAGR** vs. **\$3.11M / 11.42% CAGR**), demonstrating that a mixed model is optimal. 
3. **Didn't beat the SNP:** Despite changing the features, changing the models, and other attempts, my model still could not beat the SnP, though I think it came fairly close while avoiding hindsight bias and ensuring assumptions like trading fees remained.

### Unsuccessful Ideas
1. **Raw Return vs Cross-Sectional Rankings:** Training directly on the 90-day forward returns failed due to squared-error loss overweighting. Cross-sectional percentile rank targets resolved this stability issue.
2. **Inverse-Volatility Weighting Against Cap-Weighted Index:** Inverse-volatility weighting systematically overweights low volatility companies (Utilities, Real Estate) and underweights mega-cap tech compounders, trailing `^SP500TR` by ~4.8% annualized. Sqrt-cap weighting (`mcap_sqrt`) narrowed this gap. 
3. **Sector-Neutral Ranking with 3-Year Rolling Windows:** Computing within-sector ranks on a short 3-year rolling window produced negative Rank IC (`-0.0007` Dev, `-0.0198` Test).

### What to Try Next
- **Other features:** Experiment further with other features and see if the strategy can be improved. With hindsight bias, we know that many tech stocks out performed the SnP, but engineering features to 'choose' those was hard and quite biased. 
- **Reduce effect of old data:** Fitting the data to the entire history is probably not the most effectie way to train. The rolling window however worsened the result. Maybe some kind of moving average or some sort of weighing the data closer to the present and reducing the weight of the older data could be implemented to try and improve the training
- **Increasing Cash Holdings** My model forced the portfolio to not hold cash and instead invest in other stocks. It would be good to create some incentive for when to hold cash to avoid drawdowns. I tried to do this, but it resulted in low returns because there were months where the model would hold cash and only return 2%. 